In [2]:
import os
import sys

java_home = r"C:\Users\n.osipov\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + r"\bin;" + os.environ["PATH"]

python_path = sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("SkillBox")\
    .master("local[*]")\
    .config("spark.executor.memory", "512m")\
    .config("spark.driver.host", "127.0.0.1")\
    .config("spark.driver.bindAddress", "127.0.0.1")\
    .getOrCreate()

print(spark.version)

C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [4]:
data = [("Alice", 1), ("Bob", 2), ("Charlie", 3)]

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [7]:
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)])

In [8]:
df = spark.createDataFrame(data, schema)

In [10]:
df.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice|  1|
|    Bob|  2|
|Charlie|  3|
+-------+---+



In [17]:
df.toPandas().to_json("sales_skillbox.json", orient="records", force_ascii=False, lines=True)

C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [19]:
df_json = spark.read.json("sales_skillbox.json")

In [21]:
df_json.show()

+---+-------+
|age|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+



In [22]:
import pandas as pd

In [27]:
pdf = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie'], 'age': [1, 2, 3]})

df_pd = spark.createDataFrame(pdf)

C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [28]:
df_pd.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice|  1|
|    Bob|  2|
|Charlie|  3|
+-------+---+



In [30]:
df_pd.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [34]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

n = 1007

regions = ["Sub-Saharan Africa", "Europe", "Asia", "Middle East and North Africa", 
           "Central America and the Caribbean", "Australia and Oceania", "North America"]
countries = {
    "Sub-Saharan Africa": ["Chad", "Senegal", "Eritrea", "Cape Verde", "Mali", "Nigeria", "Kenya"],
    "Europe": ["Montenegro", "Greece", "Estonia", "Germany", "France", "Italy", "Spain"],
    "Asia": ["Japan", "China", "India", "South Korea", "Vietnam", "Thailand"],
    "Middle East and North Africa": ["Libya", "Egypt", "Morocco", "Saudi Arabia", "Turkey"],
    "Central America and the Caribbean": ["Jamaica", "Cuba", "Haiti", "Costa Rica"],
    "Australia and Oceania": ["Fiji", "Australia", "New Zealand", "Papua New Guinea"],
    "North America": ["Canada", "United States", "Mexico"]
}
item_types = ["Baby Food", "Cereal", "Clothes", "Vegetables", "Household", "Cosmetics", 
              "Beverages", "Office Supplies", "Personal Care", "Snacks", "Meat", "Fruits"]
sales_channels = ["Offline", "Online"]
priorities = ["L", "M", "H", "C"]

data = []

for i in range(n):
    region = np.random.choice(regions)
    country = np.random.choice(countries[region])
    item = np.random.choice(item_types)
    channel = np.random.choice(sales_channels)
    priority = np.random.choice(priorities)
    
    order_date = datetime(2011, 1, 1) + timedelta(days=np.random.randint(0, 2000))
    ship_date = order_date + timedelta(days=np.random.randint(1, 40))
    
    order_id = np.random.randint(100000000, 999999999)
    units_sold = np.random.randint(1, 10000)
    
    # Цены в зависимости от типа товара
    base_prices = {
        "Baby Food": (150, 90), "Cereal": (180, 110), "Clothes": (100, 35),
        "Vegetables": (140, 90), "Household": (650, 480), "Cosmetics": (420, 250),
        "Beverages": (45, 30), "Office Supplies": (620, 500), "Personal Care": (80, 55),
        "Snacks": (150, 95), "Meat": (420, 350), "Fruits": (70, 45)
    }
    unit_price, unit_cost = base_prices[item]
    unit_price *= np.random.uniform(0.9, 1.15)
    unit_cost *= np.random.uniform(0.9, 1.1)
    
    total_revenue = units_sold * unit_price
    total_cost = units_sold * unit_cost
    total_profit = total_revenue - total_cost
    
    data.append({
        "Region": region,
        "Country": country,
        "Item Type": item,
        "Sales Channel": channel,
        "Order Priority": priority,
        "Order Date": order_date.strftime("%m/%d/%Y"),
        "Order ID": order_id,
        "Ship Date": ship_date.strftime("%m/%d/%Y"),
        "Units Sold": units_sold,
        "Unit Price": round(unit_price, 2),
        "Unit Cost": round(unit_cost, 2),
        "Total Revenue": round(total_revenue, 2),
        "Total Cost": round(total_cost, 2),
        "Total Profit": round(total_profit, 2)
    })

df_hdfs = pd.DataFrame(data)

# Добавляем случайные пропуски (NaN)
for col in ["Units Sold", "Unit Price", "Unit Cost", "Total Revenue", "Total Cost", "Total Profit", "Order Priority"]:
    mask = np.random.random(n) < 0.04   # ~4% пропусков
    df_pd.loc[mask, col] = np.nan

print(f"Размер датасета: {df_pd.shape}")
print(df_pd.head(10))

Размер датасета: (1007, 14)
                              Region       Country        Item Type  \
0                      North America        Canada             Meat   
1  Central America and the Caribbean    Costa Rica  Office Supplies   
2                 Sub-Saharan Africa    Cape Verde           Snacks   
3       Middle East and North Africa  Saudi Arabia          Clothes   
4                             Europe       Germany    Personal Care   
5       Middle East and North Africa  Saudi Arabia  Office Supplies   
6              Australia and Oceania     Australia           Snacks   
7              Australia and Oceania   New Zealand  Office Supplies   
8                             Europe    Montenegro           Fruits   
9                 Sub-Saharan Africa    Cape Verde  Office Supplies   

  Sales Channel Order Priority  Order Date   Order ID   Ship Date  Units Sold  \
0        Online              L  11/10/2013  941095289  12/19/2013       467.0   
1        Online             

In [38]:
df_hdfs.to_csv("sales_skillbox.csv", index=False, encoding="utf-8-sig")
print("Файл sales_skillbox.csv успешно сохранён")

Файл sales_skillbox.csv успешно сохранён


In [60]:
df_hdfs = spark.read.csv(
    "sales_skillbox.csv",
    header=True,
    inferSchema=True,
    sep=","
)

df_hdfs.show(5)
df_hdfs.printSchema()

+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|     Country|      Item Type|Sales Channel|Order Priority|Order Date| Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|       North America|      Canada|           Meat|       Online|             L|11/10/2013|941095289|12/19/2013|       467|     388.5|   347.15|    181428.27| 162117.85|    19310.42|
|Central America a...|  Costa Rica|Office Supplies|       Online|             H|08/13/2015|454088427|08/15/2015|      6950|    558.12|   549.22|   3878938.93|3817087.03|    61851.89|
|  Sub-Saharan Africa|  Cape Verde|         Snacks|       Online|             L|01/17

In [61]:
df_hdfs.show()

+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|     Country|      Item Type|Sales Channel|Order Priority|Order Date| Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|       North America|      Canada|           Meat|       Online|             L|11/10/2013|941095289|12/19/2013|       467|     388.5|   347.15|    181428.27| 162117.85|    19310.42|
|Central America a...|  Costa Rica|Office Supplies|       Online|             H|08/13/2015|454088427|08/15/2015|      6950|    558.12|   549.22|   3878938.93|3817087.03|    61851.89|
|  Sub-Saharan Africa|  Cape Verde|         Snacks|       Online|             L|01/17

In [62]:
df_hdfs.count()

1007

In [63]:
df_hdfs = df_hdfs.drop_duplicates()

In [64]:
df_hdfs.count()

1007

In [65]:
from pyspark.sql.functions import lit

In [66]:
df_none = df.select([lit(None) for i in df_hdfs.columns])

In [67]:
df_none.show()

+----+----+----+----+----+----+----+----+----+----+----+----+----+----+
|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
+----+----+----+----+----+----+----+----+----+----+----+----+----+----+
|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|NULL|
+----+----+----+----+----+----+----+----+----+----+----+----+----+----+



In [68]:
df_hdfs = df_hdfs.union(df_none)

In [69]:
df_hdfs.count()

1010

In [70]:
df_hdfs = df_hdfs.na.drop()

In [71]:
df_hdfs.count()

1007

In [78]:
from pyspark.sql.functions import col, ceil

df_hdfs = df_hdfs.withColumn("Unit Profit", col("Unit Price") - col("Unit Cost"))

In [82]:
df_hdfs.select("Unit Price,Unit Cost,Unit Profit".split(",")).show()

+----------+---------+------------------+
|Unit Price|Unit Cost|       Unit Profit|
+----------+---------+------------------+
|     41.67|     31.8| 9.870000000000001|
|    180.85|   115.53|             65.32|
|    476.83|   270.74|206.08999999999997|
|    142.95|    91.41| 51.53999999999999|
|    638.63|   535.22|103.40999999999997|
|    395.57|   350.03| 45.54000000000002|
|    411.08|   330.72| 80.35999999999996|
|    104.34|    33.89|             70.45|
|    691.73|   466.29|            225.44|
|    172.22|    98.02|              74.2|
|    397.56|   373.62|23.939999999999998|
|    132.46|    86.32|46.140000000000015|
|      50.3|    30.08|             20.22|
|     67.56|    48.66|18.900000000000006|
|      84.0|    54.84|29.159999999999997|
|    138.42|     88.2|50.219999999999985|
|    428.77|    236.0|192.76999999999998|
|    380.27|   321.42|58.849999999999966|
|     47.61|    29.24|             18.37|
|    147.96|    85.63| 62.33000000000001|
+----------+---------+------------

In [83]:
df_hdfs.groupby("Region").sum("Units Sold").show()

+--------------------+---------------+
|              Region|sum(Units Sold)|
+--------------------+---------------+
|Middle East and N...|         732290|
|Australia and Oce...|         665827|
|              Europe|         666616|
|  Sub-Saharan Africa|         678120|
|Central America a...|         663901|
|       North America|         823011|
|                Asia|         783535|
+--------------------+---------------+



In [84]:
df_hdfs.groupby("Region").avg("Units Sold").show()

+--------------------+-----------------+
|              Region|  avg(Units Sold)|
+--------------------+-----------------+
|Middle East and N...|5050.275862068966|
|Australia and Oce...|5044.143939393939|
|              Europe|4937.896296296296|
|  Sub-Saharan Africa|4878.561151079137|
|Central America a...|          4742.15|
|       North America|5049.147239263803|
|                Asia|5121.143790849673|
+--------------------+-----------------+



In [86]:
from pyspark.sql.functions import sum, avg

df_hdfs.groupby("Country").agg(sum("Units Sold").alias("Units Sold Sum"), avg("Units Sold").alias("Units Sold Avg")).show()

+-------------+--------------+------------------+
|      Country|Units Sold Sum|    Units Sold Avg|
+-------------+--------------+------------------+
|         Chad|         51058|            3647.0|
|      Senegal|         62870| 5239.166666666667|
|      Eritrea|        133304| 6059.272727272727|
|         Fiji|        151331| 5404.678571428572|
|       Turkey|        105258| 5012.285714285715|
|      Germany|        127336| 5305.666666666667|
|       France|         66030|5079.2307692307695|
|       Greece|        109352| 4970.545454545455|
|United States|        306073| 6001.431372549019|
|        India|        151964| 5427.285714285715|
|        China|        128653| 4764.925925925926|
|      Nigeria|        105891| 5042.428571428572|
|        Italy|        109956|5787.1578947368425|
|        Spain|         70247|3902.6111111111113|
|         Cuba|        175039| 5001.114285714286|
|     Thailand|        142975| 4930.172413793103|
|      Morocco|        104564| 4979.238095238095|


In [88]:
df_group_by = df_hdfs.groupby("Country").agg(sum("Units Sold").alias("Units Sold Sum"), avg("Units Sold").alias("Units Sold Avg"))

In [89]:
df_hdfs = df_hdfs.join(df_group_by, "Country", "left")

In [91]:
df_hdfs.show()

+----------------+--------------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+------------------+--------------+-----------------+
|         Country|              Region|      Item Type|Sales Channel|Order Priority|Order Date| Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|       Unit Profit|Units Sold Sum|   Units Sold Avg|
+----------------+--------------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+------------------+--------------+-----------------+
|           Egypt|Middle East and N...|      Beverages|      Offline|             C|04/12/2013|942299350|04/15/2013|      8520|     41.67|     31.8|    355038.81| 270931.71|     84107.1| 9.870000000000001|        180615|4753.026315789473|
|          Greece|              Europe|     

In [97]:
df_hdfs.rdd.getNumPartitions()

10

In [94]:
df_hdfs = df_hdfs.repartition(300)

In [96]:
df_hdfs = df_hdfs.coalesce(10)

In [102]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, FloatType

In [103]:
def my_udf(column):
    return str(column).upper()

In [104]:
my_udf('Anton')

'ANTON'

In [105]:
my_udf = udf(my_udf, StringType())

C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [106]:
df_hdfs = df_hdfs.withColumn("Upper Country", my_udf(col("Country")))

In [109]:
df_hdfs.select("Country,Region,Upper Country".split(",")).show()

+----------------+--------------------+----------------+
|         Country|              Region|   Upper Country|
+----------------+--------------------+----------------+
|     New Zealand|Australia and Oce...|     NEW ZEALAND|
|Papua New Guinea|Australia and Oce...|PAPUA NEW GUINEA|
|           Libya|Middle East and N...|           LIBYA|
|     New Zealand|Australia and Oce...|     NEW ZEALAND|
|Papua New Guinea|Australia and Oce...|PAPUA NEW GUINEA|
|           Libya|Middle East and N...|           LIBYA|
|      Cape Verde|  Sub-Saharan Africa|      CAPE VERDE|
|Papua New Guinea|Australia and Oce...|PAPUA NEW GUINEA|
|           Libya|Middle East and N...|           LIBYA|
|      Cape Verde|  Sub-Saharan Africa|      CAPE VERDE|
|Papua New Guinea|Australia and Oce...|PAPUA NEW GUINEA|
|           Libya|Middle East and N...|           LIBYA|
|      Cape Verde|  Sub-Saharan Africa|      CAPE VERDE|
|         Eritrea|  Sub-Saharan Africa|         ERITREA|
|           Japan|             

In [112]:
def my_udf(column_1, column_2):
    from math import ceil
    res = ceil((column_1 - column_2) * 100) / 100
    return res

In [113]:
my_udf(4, 2)

2.0

In [114]:
my_udf = udf(my_udf, FloatType())

C:\Users\n.osipov\AppData\Local\miniconda3\envs\arrow_env\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [115]:
df_hdfs = df_hdfs.withColumn("UDF Unit Profit", my_udf(col("Unit Price"), col("Unit Cost")))

In [120]:
df_hdfs.select("UDF Unit Profit,Region".split(",")).show()

+---------------+--------------------+
|UDF Unit Profit|              Region|
+---------------+--------------------+
|         147.12|       North America|
|           12.4|Middle East and N...|
|         181.75|  Sub-Saharan Africa|
|          81.56|  Sub-Saharan Africa|
|          18.25|Middle East and N...|
|          77.28|  Sub-Saharan Africa|
|          52.49|Central America a...|
|          75.91|                Asia|
|          53.83|Central America a...|
|          40.84|Central America a...|
|          58.85|              Europe|
|          134.4|Middle East and N...|
|          93.37|                Asia|
|          157.5|Australia and Oce...|
|          31.24|                Asia|
|         157.87|Middle East and N...|
|          58.85|Central America a...|
|           57.8|       North America|
|           15.5|       North America|
|          37.36|Australia and Oce...|
+---------------+--------------------+
only showing top 20 rows


In [121]:
df_hdfs.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [Country#554, Region#553, Item Type#555, Sales Channel#556, Order Priority#557, Order Date#558, Order ID#559, Ship Date#560, Units Sold#561, Unit Price#562, Unit Cost#563, Total Revenue#564, Total Cost#565, Total Profit#566, Unit Profit#868, Units Sold Sum#1178L, Units Sold Avg#1179, pythonUDF0#1750 AS Upper Country#1580, pythonUDF1#1751 AS UDF Unit Profit#1707]
   +- ArrowEvalPython [my_udf(Country#554)#1579, my_udf(Unit Price#562, Unit Cost#563)#1706], [pythonUDF0#1750, pythonUDF1#1751], 101
      +- Coalesce 10
         +- Exchange RoundRobinPartitioning(300), REPARTITION_BY_NUM, [plan_id=4513]
            +- Project [Country#554, Region#553, Item Type#555, Sales Channel#556, Order Priority#557, Order Date#558, Order ID#559, Ship Date#560, Units Sold#561, Unit Price#562, Unit Cost#563, Total Revenue#564, Total Cost#565, Total Profit#566, Unit Profit#868, Units Sold Sum#1178L, Units Sold Avg#1179]
               +- Br

In [122]:
df_hdfs_new = spark.read.csv(
    "sales_skillbox.csv",
    header=True,
    inferSchema=True,
    sep=","
)

In [125]:
df_hdfs_new.explain()

== Physical Plan ==
FileScan csv [Region#1769,Country#1770,Item Type#1771,Sales Channel#1772,Order Priority#1773,Order Date#1774,Order ID#1775,Ship Date#1776,Units Sold#1777,Unit Price#1778,Unit Cost#1779,Total Revenue#1780,Total Cost#1781,Total Profit#1782] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/n.osipov/Профессия Data Scientist с нуля до PRO. Тариф ..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Region:string,Country:string,Item Type:string,Sales Channel:string,Order Priority:string,O...




In [128]:
df_hdfs_new.filter(col("Unit Price") >= 100).explain()

== Physical Plan ==
*(1) Filter (isnotnull(Unit Price#1778) AND (Unit Price#1778 >= 100.0))
+- FileScan csv [Region#1769,Country#1770,Item Type#1771,Sales Channel#1772,Order Priority#1773,Order Date#1774,Order ID#1775,Ship Date#1776,Units Sold#1777,Unit Price#1778,Unit Cost#1779,Total Revenue#1780,Total Cost#1781,Total Profit#1782] Batched: false, DataFilters: [isnotnull(Unit Price#1778), (Unit Price#1778 >= 100.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/n.osipov/Профессия Data Scientist с нуля до PRO. Тариф ..., PartitionFilters: [], PushedFilters: [IsNotNull(Unit Price), GreaterThanOrEqual(Unit Price,100.0)], ReadSchema: struct<Region:string,Country:string,Item Type:string,Sales Channel:string,Order Priority:string,O...




In [129]:
df_hdfs_new.printSchema()

root
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Item Type: string (nullable = true)
 |-- Sales Channel: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order ID: integer (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Unit Cost: double (nullable = true)
 |-- Total Revenue: double (nullable = true)
 |-- Total Cost: double (nullable = true)
 |-- Total Profit: double (nullable = true)



In [130]:
df_hdfs_new.show()

+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|     Country|      Item Type|Sales Channel|Order Priority|Order Date| Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|       North America|      Canada|           Meat|       Online|             L|11/10/2013|941095289|12/19/2013|       467|     388.5|   347.15|    181428.27| 162117.85|    19310.42|
|Central America a...|  Costa Rica|Office Supplies|       Online|             H|08/13/2015|454088427|08/15/2015|      6950|    558.12|   549.22|   3878938.93|3817087.03|    61851.89|
|  Sub-Saharan Africa|  Cape Verde|         Snacks|       Online|             L|01/17

In [140]:
from pyspark.sql.functions import coalesce, col
df_hdfs_new = df_hdfs_new.withColumn("Region", coalesce(col("Region"), lit("Earth")))

In [141]:
df_hdfs_new.filter(col("Region") == "Earth").show()

+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+
|Region|Country|Item Type|Sales Channel|Order Priority|Order Date|Order ID|Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+
+------+-------+---------+-------------+--------------+----------+--------+---------+----------+----------+---------+-------------+----------+------------+



In [145]:
df_hdfs_new = df_hdfs_new.cache()

In [143]:
df_hdfs_new.show()

+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|              Region|     Country|      Item Type|Sales Channel|Order Priority|Order Date| Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|
+--------------------+------------+---------------+-------------+--------------+----------+---------+----------+----------+----------+---------+-------------+----------+------------+
|       North America|      Canada|           Meat|       Online|             L|11/10/2013|941095289|12/19/2013|       467|     388.5|   347.15|    181428.27| 162117.85|    19310.42|
|Central America a...|  Costa Rica|Office Supplies|       Online|             H|08/13/2015|454088427|08/15/2015|      6950|    558.12|   549.22|   3878938.93|3817087.03|    61851.89|
|  Sub-Saharan Africa|  Cape Verde|         Snacks|       Online|             L|01/17

In [144]:
df_hdfs_new.explain()

== Physical Plan ==
InMemoryTableScan [Region#1958, Country#1770, Item Type#1771, Sales Channel#1772, Order Priority#1773, Order Date#1774, Order ID#1775, Ship Date#1776, Units Sold#1777, Unit Price#1778, Unit Cost#1779, Total Revenue#1780, Total Cost#1781, Total Profit#1782]
   +- InMemoryRelation [Region#1958, Country#1770, Item Type#1771, Sales Channel#1772, Order Priority#1773, Order Date#1774, Order ID#1775, Ship Date#1776, Units Sold#1777, Unit Price#1778, Unit Cost#1779, Total Revenue#1780, Total Cost#1781, Total Profit#1782], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Project [coalesce(Region#1769, Earth) AS Region#1958, Country#1770, Item Type#1771, Sales Channel#1772, Order Priority#1773, Order Date#1774, Order ID#1775, Ship Date#1776, Units Sold#1777, Unit Price#1778, Unit Cost#1779, Total Revenue#1780, Total Cost#1781, Total Profit#1782]
            +- FileScan csv [Region#1769,Country#1770,Item Type#1771,Sales Channel#1772,Order Priority#1773,Ord

In [146]:
df_hdfs_new.unpersist()

DataFrame[Region: string, Country: string, Item Type: string, Sales Channel: string, Order Priority: string, Order Date: string, Order ID: int, Ship Date: string, Units Sold: int, Unit Price: double, Unit Cost: double, Total Revenue: double, Total Cost: double, Total Profit: double]